In [ ]:
# Google Drive Upload

## Setup Instructions:
1. Install required packages: `pip install pydrive2`
2. Go to [Google Cloud Console](https://console.cloud.google.com/)
3. Create a new project or select existing one
4. Enable Google Drive API
5. Create OAuth 2.0 credentials (Desktop app)
6. Download credentials as `client_secrets.json` and place in this directory

In [ ]:
# Install required package (run once)
# !pip install pydrive2

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
import os
from tqdm import tqdm

In [ ]:
# Authenticate and create GoogleDrive instance
gauth = GoogleAuth()
gauth.LocalWebserverAuth()  # Creates local webserver and automatically handles authentication
drive = GoogleDrive(gauth)
print("Authentication successful!")

In [ ]:
def create_folder(drive, folder_name, parent_id=None):
    """Create a folder in Google Drive"""
    folder_metadata = {
        'title': folder_name,
        'mimeType': 'application/vnd.google-apps.folder'
    }
    if parent_id:
        folder_metadata['parents'] = [{'id': parent_id}]
    
    folder = drive.CreateFile(folder_metadata)
    folder.Upload()
    return folder['id']


def upload_file(drive, file_path, parent_id=None):
    """Upload a single file to Google Drive"""
    file_name = os.path.basename(file_path)
    file_metadata = {'title': file_name}
    if parent_id:
        file_metadata['parents'] = [{'id': parent_id}]
    
    file = drive.CreateFile(file_metadata)
    file.SetContentFile(file_path)
    file.Upload()
    return file['id']


def upload_folder(drive, local_folder_path, parent_id=None, root_name=None):
    """
    Recursively upload a folder and its contents to Google Drive
    
    Args:
        drive: GoogleDrive instance
        local_folder_path: Path to local folder to upload
        parent_id: Google Drive folder ID to upload into (None for root)
        root_name: Name for the root folder (uses folder name if None)
    """
    folder_name = root_name if root_name else os.path.basename(local_folder_path)
    
    # Create the folder in Google Drive
    folder_id = create_folder(drive, folder_name, parent_id)
    print(f"Created folder: {folder_name} (ID: {folder_id})")
    
    # Get all items in the local folder
    items = os.listdir(local_folder_path)
    
    for item in tqdm(items, desc=f"Uploading {folder_name}"):
        item_path = os.path.join(local_folder_path, item)
        
        if os.path.isdir(item_path):
            # Recursively upload subfolder
            upload_folder(drive, item_path, folder_id)
        else:
            # Upload file
            try:
                upload_file(drive, item_path, folder_id)
                # print(f"  Uploaded: {item}")
            except Exception as e:
                print(f"  ❌ Failed to upload {item}: {e}")
    
    return folder_id

## Upload Folders to Google Drive

Specify the folders you want to upload below:

In [ ]:
# Example: Upload sites_imputed folder
folder_to_upload = "CPCB/sites_imputed"
folder_id = upload_folder(drive, folder_to_upload, root_name="sites_imputed")
print(f"\n✅ Upload complete! Folder ID: {folder_id}")

In [ ]:
# Upload multiple folders
folders_to_upload = [
    "CPCB/sites_comb",
    "CPCB/sites_comb_max",
    "sites_comb_max_t",
    # Add more folders as needed
]

uploaded_folders = {}

for folder_path in folders_to_upload:
    if os.path.exists(folder_path):
        print(f"\n{'='*60}")
        print(f"Uploading: {folder_path}")
        print(f"{'='*60}")
        try:
            folder_name = os.path.basename(folder_path)
            folder_id = upload_folder(drive, folder_path, root_name=folder_name)
            uploaded_folders[folder_path] = folder_id
            print(f"✅ {folder_path} uploaded successfully!")
        except Exception as e:
            print(f"❌ Failed to upload {folder_path}: {e}")
    else:
        print(f"⚠️  Folder not found: {folder_path}")

print(f"\n{'='*60}")
print(f"Upload Summary: {len(uploaded_folders)}/{len(folders_to_upload)} folders uploaded")
print(f"{'='*60}")

## Helper Functions

Check what's in your Google Drive:

In [ ]:
# List folders in your Google Drive root
file_list = drive.ListFile({'q': "'root' in parents and trashed=false"}).GetList()

print("Folders and files in Google Drive root:")
print("="*60)
for file in file_list:
    file_type = "📁" if file['mimeType'] == 'application/vnd.google-apps.folder' else "📄"
    print(f"{file_type} {file['title']} (ID: {file['id']})")

## Notes

- **First time setup**: You'll need to create OAuth credentials from Google Cloud Console
- **Authentication**: A browser window will open for authentication on first run
- **Large folders**: Upload progress is shown with tqdm progress bars
- **Error handling**: Failed uploads are logged but don't stop the entire process
- **Resuming**: If interrupted, rerun the cell - PyDrive2 handles duplicates by creating new files (you may want to delete old versions)